# Task 2A: Hyperparameter Experiments

Greedy search: change one parameter at a time, carry the best forward.

**Baseline**: fixed chunking, 1024 tokens, overlap 200, top_k=5, alpha=0.5, no reranking, e5-large

**Experiments**:
1. Chunk size: 512 vs 1024 vs 2048
2. Chunk overlap: 100 vs 200 vs 400
3. Chunking strategy: fixed vs recursive vs layout_aware
4. Top-K: 3 vs 5 vs 10
5. Alpha (hybrid weight): 0.3 vs 0.5 vs 0.7 vs 1.0
6. Reranking: on vs off

In [1]:
import sys
sys.path.insert(0, "..")

import json
from src.parsing import parse_all_pdfs
from src.pipeline import RAGPipeline
from src.evaluation import load_golden_dataset, run_experiment, results_to_dataframe
from src.config import DEFAULT_CONFIG

In [2]:
parsed_texts = parse_all_pdfs(use_cache=True)
golden = load_golden_dataset("../data/golden_dataset.json")
print(f"Loaded {len(golden)} golden QA pairs")

Loaded 30 golden QA pairs


In [3]:
all_results = []

def run_exp(name: str, config_overrides: dict, parsed_texts=parsed_texts, golden=golden):
    """Helper: build pipeline, ingest, evaluate, store result."""
    config = {**DEFAULT_CONFIG, **config_overrides}
    config["collection_name"] = f"exp_{name.replace(' ', '_').lower()}"
    pipeline = RAGPipeline(config)
    pipeline.ingest(parsed_texts)
    result = run_experiment(pipeline, golden, name)
    all_results.append(result)
    print(f"\n{name}:")
    for metric in ["faithfulness", "answer_relevancy", "context_recall", "context_precision"]:
        print(f"  {metric}: {result.get(metric, 'N/A'):.4f}")
    return result

## Experiment 0: Baseline

In [4]:
baseline = run_exp("Baseline", {
    "chunking_strategy": "fixed",
    "chunk_size": 1024,
    "chunk_overlap": 200,
    "top_k": 5,
    "alpha": 0.5,
    "use_reranking": False,
})

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Baseline:
  faithfulness: 0.9514
  answer_relevancy: 0.8354
  context_recall: 0.8667
  context_precision: 0.7824


## Experiment 1: Chunk Size

In [5]:
for size in [512, 2048]:
    run_exp(f"Chunk size {size}", {
        "chunk_size": size,
        "chunk_overlap": 200,
        "chunking_strategy": "fixed",
        "top_k": 5,
        "alpha": 0.5,
        "use_reranking": False,
    })

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Chunk size 512:
  faithfulness: 0.8667
  answer_relevancy: 0.7645
  context_recall: 0.7667
  context_precision: 0.6859


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Chunk size 2048:
  faithfulness: 0.9217
  answer_relevancy: 0.9312
  context_recall: 0.9667
  context_precision: 0.7385


## Experiment 2: Chunk Overlap
Using best chunk_size from Experiment 1.

In [6]:
# NOTE: Update best_chunk_size based on Experiment 1 results
best_chunk_size = 1024  # Update after running Experiment 1

for overlap in [100, 400]:
    run_exp(f"Overlap {overlap}", {
        "chunk_size": best_chunk_size,
        "chunk_overlap": overlap,
        "chunking_strategy": "fixed",
        "top_k": 5,
        "alpha": 0.5,
        "use_reranking": False,
    })

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Overlap 100:
  faithfulness: 0.9522
  answer_relevancy: 0.8137
  context_recall: 0.8667
  context_precision: 0.6370


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Overlap 400:
  faithfulness: 0.8536
  answer_relevancy: 0.8164
  context_recall: 0.8667
  context_precision: 0.7350


## Experiment 3: Chunking Strategy
Using best chunk_size and overlap from above.

In [7]:
best_overlap = 200  # Update after running Experiment 2

for strategy in ["recursive", "layout_aware"]:
    run_exp(f"Strategy {strategy}", {
        "chunk_size": best_chunk_size,
        "chunk_overlap": best_overlap,
        "chunking_strategy": strategy,
        "top_k": 5,
        "alpha": 0.5,
        "use_reranking": False,
    })

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Strategy recursive:
  faithfulness: 0.9514
  answer_relevancy: 0.8381
  context_recall: 0.8667
  context_precision: 0.7798


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Strategy layout_aware:
  faithfulness: 0.8860
  answer_relevancy: 0.6366
  context_recall: 0.5667
  context_precision: 0.3378


## Experiment 4: Top-K

In [8]:
best_strategy = "fixed"  # Update after running Experiment 3

for k in [3, 10]:
    run_exp(f"Top-K {k}", {
        "chunk_size": best_chunk_size,
        "chunk_overlap": best_overlap,
        "chunking_strategy": best_strategy,
        "top_k": k,
        "alpha": 0.5,
        "use_reranking": False,
    })

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Top-K 3:
  faithfulness: 0.9230
  answer_relevancy: 0.8743
  context_recall: 0.8667
  context_precision: 0.8333


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised


Top-K 10:
  faithfulness: 0.8522
  answer_relevancy: 0.8992
  context_recall: 0.9333
  context_precision: 0.7356


## Experiment 5: Alpha (Hybrid Weight)

In [9]:
best_top_k = 5  # Update after running Experiment 4

for alpha in [0.3, 0.7, 1.0]:
    run_exp(f"Alpha {alpha}", {
        "chunk_size": best_chunk_size,
        "chunk_overlap": best_overlap,
        "chunking_strategy": best_strategy,
        "top_k": best_top_k,
        "alpha": alpha,
        "use_reranking": False,
    })

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Alpha 0.3:
  faithfulness: 0.9452
  answer_relevancy: 0.7440
  context_recall: 0.7667
  context_precision: 0.7142


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Alpha 0.7:
  faithfulness: 0.9429
  answer_relevancy: 0.8330
  context_recall: 0.8667
  context_precision: 0.7639


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


Alpha 1.0:
  faithfulness: 0.8600
  answer_relevancy: 0.8997
  context_recall: 0.9333
  context_precision: 0.8201


## Experiment 6: Reranking

In [10]:
import warnings                                                                                                
import logging             

warnings.filterwarnings("ignore")                                                                              
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)   

best_alpha = 0.5  # Update after running Experiment 5

run_exp("With Reranking", {
    "chunk_size": best_chunk_size,
    "chunk_overlap": best_overlap,
    "chunking_strategy": best_strategy,
    "top_k": best_top_k,
    "alpha": best_alpha,
    "use_reranking": True,
})

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 g


With Reranking:
  faithfulness: 0.9600
  answer_relevancy: 0.8741
  context_recall: 0.8667
  context_precision: 0.8094


{'experiment': 'With Reranking',
 'config': {'chunk_size': 1024,
  'chunk_overlap': 200,
  'chunking_strategy': 'fixed',
  'embedding_model': 'intfloat/multilingual-e5-large',
  'top_k': 5,
  'alpha': 0.5,
  'use_reranking': True,
  'reranker_model': 'BAAI/bge-reranker-v2-m3',
  'use_query_rewriting': False,
  'llm_model': 'gpt-4o-mini',
  'collection_name': 'exp_with_reranking'},
 'faithfulness': np.float64(0.9599999999999999),
 'answer_relevancy': np.float64(0.8740755013883373),
 'context_recall': np.float64(0.8666666666666667),
 'context_precision': np.float64(0.8094444443932824)}

## Summary Table

In [11]:
df = results_to_dataframe(all_results)
print(df.to_string(index=False))
df

           Experiment  Faithfulness  Answer Relevancy  Context Recall  Context Precision
             Baseline      0.951389          0.835360        0.866667           0.782407
       Chunk size 512      0.866667          0.764482        0.766667           0.685926
      Chunk size 2048      0.921746          0.931188        0.966667           0.738519
          Overlap 100      0.952183          0.813737        0.866667           0.637037
          Overlap 400      0.853571          0.816426        0.866667           0.735046
   Strategy recursive      0.951389          0.838146        0.866667           0.779815
Strategy layout_aware      0.885979          0.636562        0.566667           0.337778
              Top-K 3      0.923016          0.874254        0.866667           0.833333
             Top-K 10      0.852222          0.899230        0.933333           0.735554
            Alpha 0.3      0.945238          0.743993        0.766667           0.714213
            Alpha 0.7

,Experiment,Faithfulness,Answer Relevancy,Context Recall,Context Precision
0,Baseline,0.951389,0.835360,0.866667,0.782407
1,Chunk size 512,0.866667,0.764482,0.766667,0.685926
2,Chunk size 2048,0.921746,0.931188,0.966667,0.738519
3,Overlap 100,0.952183,0.813737,0.866667,0.637037
4,Overlap 400,0.853571,0.816426,0.866667,0.735046
5,Strategy recursive,0.951389,0.838146,0.866667,0.779815
6,Strategy layout_aware,0.885979,0.636562,0.566667,0.337778
7,Top-K 3,0.923016,0.874254,0.866667,0.833333
8,Top-K 10,0.852222,0.899230,0.933333,0.735554
9,Alpha 0.3,0.945238,0.743993,0.766667,0.714213


In [12]:
import json

def make_serializable(r):
    return {k: float(v) if isinstance(v, float) else v for k, v in r.items() if k != "config"}

with open("../data/experiment_results.json", "w", encoding="utf-8") as f:
    json.dump([make_serializable(r) for r in all_results], f, ensure_ascii=False, indent=2)
print("Results saved to data/experiment_results.json")

Results saved to data/experiment_results.json
